In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila


In [3]:

# flybase/Flybase_Droso_Gene_Phenotype.csv
# Monarch/Drosophila/Gene_Droso_Phenotype_MONARCH.csv


In [4]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Drosophila_gene_phenotype
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Drosophila_gene_phenotype/Drosophila_gene_phenotype.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

In [5]:
#

In [6]:
droso = pd.read_csv(
    f'{BASE_DIR}data_collection/databases_for_mapping/ncbi/Drosophila_melanogaster.gene_info',
    sep='\t',
    
)
droso["LocusTag"] = droso["LocusTag"].str.replace("Dmel_", "", regex=False)
droso = droso[droso['LocusTag'] != '-']
droso_symbol2Locus_dict = dict(zip(droso['Symbol'], droso['LocusTag']))
droso_symbol2Desc_dict = dict(zip(droso['Symbol'], droso['description']))
droso_locus2Desc_dict = dict(zip(droso['LocusTag'], droso['description']))


droso
# droso_symbol2Locus_dict

,#tax_id,GeneID,Symbol,LocusTag,Synonyms,dbXrefs,chromosome,map_location,description,type_of_gene,Symbol_from_nomenclature_authority,Full_name_from_nomenclature_authority,Nomenclature_status,Other_designations,Modification_date,Feature_type
0,7227,30970,CG3038,CG3038,Dmel\CG3038|EG:BACR37P7.1|GalT3|dbeta3GnT,FLYBASE:FBgn0040373|AllianceGenome:FB:FBgn0040373,X,1A1-1A1|1-0 cM,uncharacterized protein,protein-coding,CG3038,uncharacterized protein,O,uncharacterized protein|CG3038-PA|CG3038-PB|CG...,20250308,-
1,7227,30971,G9a,CG2995,CG2995|Dmel\CG2995|EG:BACR37P7.2|EHMT|G9ae|cab...,FLYBASE:FBgn0040372|AllianceGenome:FB:FBgn0040372,X,1A1-1A1|1-0 cM,G9a,protein-coding,G9a,G9a,O,G9a|CG2995-PA|CG2995-PB|G9a-PA|G9a-PB|dG9a|euc...,20250308,-
2,7227,30972,CG13377,CG13377,BG01596|Dmel\CG13377|EG:BACR37P7.9,FLYBASE:FBgn0261446|AllianceGenome:FB:FBgn0261446,X,1A1-1A1|1-0 cM,uncharacterized protein,protein-coding,CG13377,uncharacterized protein,O,uncharacterized protein|BG01596|CG13377-PA|CG1...,20250308,-
3,7227,30973,cin,CG2945,CG2945|CIN|Dmel\CG2945|EG:BACR37P7.3|fs(1)M50|...,FLYBASE:FBgn0000316|AllianceGenome:FB:FBgn0000316,X,1A1-1A1|1-0 cM,cinnamon,protein-coding,cin,cinnamon,O,cinnamon|CG2945-PA|CG2945-PB|cin-PA|cin-PB,20250308,-
4,7227,30975,ewg,CG3114,CG3114|Dmel\CG3114|EC3|EG:BACR37P7.7|EWG|Ewg|N...,FLYBASE:FBgn0005427|AllianceGenome:FB:FBgn0005427,X,1A1-1A1|1-0 cM,erect wing,protein-coding,ewg,erect wing,O,erect wing|CG3114-PA|CG3114-PB|CG3114-PC|CG311...,20250308,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25074,7227,86703977,lncRNA:CR46531,CR46531,CR46531|Dmel\CR46531,FLYBASE:FBgn0289440|AllianceGenome:FB:FBgn0289440,3R,88A8-88A8|3-54 cM,long non-coding RNA:CR46531,ncRNA,lncRNA:CR46531,long non-coding RNA:CR46531,O,-,20241028,-
25075,7227,86703978,lncRNA:CR46532,CR46532,CR46532|Dmel\CR46532,FLYBASE:FBgn0289441|AllianceGenome:FB:FBgn0289441,3R,84D5-84D5|3-48 cM,long non-coding RNA:CR46532,ncRNA,lncRNA:CR46532,long non-coding RNA:CR46532,O,-,20241028,-
25076,7227,86703979,lncRNA:CR46533,CR46533,CR46533|Dmel\CR46533,FLYBASE:FBgn0289442|AllianceGenome:FB:FBgn0289442,X,8D6-8D6|1-27 cM,long non-coding RNA:CR46533,ncRNA,lncRNA:CR46533,long non-coding RNA:CR46533,O,-,20241028,-
25077,7227,86703980,lncRNA:CR46534,CR46534,CR46534|Dmel\CR46534,FLYBASE:FBgn0289443|AllianceGenome:FB:FBgn0289443,2R,56F8-56F8|2-90 cM,long non-coding RNA:CR46534,ncRNA,lncRNA:CR46534,long non-coding RNA:CR46534,O,-,20241028,-


# flybase

In [7]:
flybase = pd.read_csv(PROC_DIR + 'flybase/Flybase_Droso_Gene_Phenotype.csv')
flybase.columns = flybase.columns.str.lower()
flybase['head'] = flybase['head'].map(droso_symbol2Locus_dict).fillna(flybase['head'])
flybase['kg_type'] = 'Generalised'
flybase['species'] = 'D.melanogaster'
print(f"flybase: {len(flybase):,} rows")
flybase['head_id_is'] = 'NCBI_ID'
flybase['tail_id_is'] = 'Quick_GO'
flybase['head_detail_name'] = flybase['head'].map(droso_symbol2Desc_dict).fillna(flybase['head'].map(droso_locus2Desc_dict))
flybase['tail_detail_name'] = flybase['tail']
flybase = flybase[~flybase['head_detail_name'].isna()]
flybase

flybase: 115,011 rows


,head,relation,tail,head_type,tail_type,kg_source,head_id_is,tail_id_is,kg_type,species,head_detail_name,tail_detail_name
11,CG31196,Gene_Phenotype,fertile,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,14-3-3epsilon,fertile
12,CG31196,Gene_Phenotype,viable,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,14-3-3epsilon,viable
13,CG31196,Gene_Phenotype,abnormal neuroanatomy,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,14-3-3epsilon,abnormal neuroanatomy
14,CG31196,Gene_Phenotype,female fertile,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,14-3-3epsilon,female fertile
15,CG31196,Gene_Phenotype,larval intersegmental nerve branch ISNb of A1-7,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,14-3-3epsilon,larval intersegmental nerve branch ISNb of A1-7
...,...,...,...,...,...,...,...,...,...,...,...,...
115002,CG12314,Gene_Phenotype,karyosome,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,zucchini,karyosome
115007,CG2893,Gene_Phenotype,abnormal neurophysiology,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,zydeco,abnormal neurophysiology
115008,CG2893,Gene_Phenotype,viable,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,zydeco,viable
115009,CG2893,Gene_Phenotype,bang sensitive,Gene,Phenotype,FlyBase,NCBI_ID,Quick_GO,Generalised,D.melanogaster,zydeco,bang sensitive


# Consolidate into Unified Schem

In [8]:
# List all source DataFrames to include
source_dfs = [
    flybase
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 76,324


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,CG31196,Gene_Phenotype,fertile,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,14-3-3epsilon,fertile,D.melanogaster
1,CG31196,Gene_Phenotype,viable,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,14-3-3epsilon,viable,D.melanogaster
2,CG31196,Gene_Phenotype,abnormal neuroanatomy,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,14-3-3epsilon,abnormal neuroanatomy,D.melanogaster
3,CG31196,Gene_Phenotype,female fertile,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,14-3-3epsilon,female fertile,D.melanogaster
4,CG31196,Gene_Phenotype,larval intersegmental nerve branch ISNb of A1-7,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,14-3-3epsilon,larval intersegmental nerve branch ISNb of A1-7,D.melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...
76319,CG12314,Gene_Phenotype,karyosome,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,zucchini,karyosome,D.melanogaster
76320,CG2893,Gene_Phenotype,abnormal neurophysiology,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,zydeco,abnormal neurophysiology,D.melanogaster
76321,CG2893,Gene_Phenotype,viable,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,zydeco,viable,D.melanogaster
76322,CG2893,Gene_Phenotype,bang sensitive,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,zydeco,bang sensitive,D.melanogaster


# Sanity Check — Distinct Values

In [9]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Gene_Phenotype'}
head_type           : {'Gene'}
tail_type           : {'Phenotype'}
relation_type       : {None}
kg_source           : {'FlyBase'}
head_id_is          : {'NCBI_ID'}
tail_id_is          : {'Quick_GO'}


In [10]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 21 unresolvable rows → 76,303 remaining


# NaN Audit (pre-dedup)

In [11]:
true_nan   = final_df.isna().sum()
genDR_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_genDR_count": genDR_nan,
    'Total_NaN_like':     true_nan + genDR_nan
})

,NaN_count,'NAN'_genDR_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,76303,0,76303
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [12]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 76,303  |  After dedup: 76,303


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,CG10001,Gene_Phenotype,abnormal sleep,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,Allatostatin A receptor 2,abnormal sleep
1,CG10001,Gene_Phenotype,decreased rate of movement,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,Allatostatin A receptor 2,decreased rate of movement
2,CG10001,Gene_Phenotype,long lived,Gene,None,Phenotype,FlyBase,Generalised,NCBI_ID,Quick_GO,Allatostatin A receptor 2,long lived


In [13]:
true_nan   = final_df_dedup.isna().sum()
genDR_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_genDR_count": genDR_nan,
    'Total_NaN_like':     true_nan + genDR_nan
})

,NaN_count,'NAN'_genDR_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,76303,0,76303
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [14]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'FlyBase'} kg_source
FlyBase    76303
Name: count, dtype: int64


In [15]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    76303
Name: count, dtype: int64


In [16]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 76,303 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Drosophila/Drosophila_gene_phenotype/Drosophila_gene_phenotype.csv


In [18]:
#